# ETL

In [1]:
# imports necesarios
import pandas as pd
import os

In [2]:
print("Directorio de trabajo:", os.getcwd())


Directorio de trabajo: c:\PROYECTOS\SQL\homicidios\src


In [3]:
os.chdir('c:\\PROYECTOS\\SQL\\homicidios\\data')

In [4]:
df = pd.read_csv(
    'c:\\PROYECTOS\\SQL\\homicidios\\data\\SAT-HD-BU.csv',
    sep=',',
    encoding='utf-8',
    low_memory=False
)

In [5]:
df.head()

,id_hecho,tipo_persona_id,tipo_persona,cant_inc,cant_vic,federal,provincia_id,provincia_nombre,departamento_id,departamento_nombre,...,victi_situacion_ocupacional_otro,victima_relacion_inculpado,inculpado_sexo,inculpado_identidad_genero,inculpado_identidad_genero_otro,inculpado_tr_edad,inculpado_18_años_o_mas,inculpado_clase,inculpado_otro_clase,inculpado_relacion_victima
0,11535,Víctima idRegistro 12235,Víctima,3,1,No,58,Neuquén,58035,Confluencia,...,NaN,Sin relación,No corresponde,No corresponde,-------,No corresponde,No corresponde,No corresponde,----,No corresponde
1,11535,Imputado idRegistro 12929,Imputado,3,1,No,58,Neuquén,58035,Confluencia,...,---------,No corresponde,Masculino,Varón,NaN,Sin determinar,Sin determinar,Sin determinación,NaN,Sin relación
2,11535,Imputado idRegistro 12931,Imputado,3,1,No,58,Neuquén,58035,Confluencia,...,---------,No corresponde,Masculino,Varón,NaN,Sin determinar,Sin determinar,Sin determinación,NaN,Sin relación
3,11535,Imputado idRegistro 12930,Imputado,3,1,No,58,Neuquén,58035,Confluencia,...,---------,No corresponde,Masculino,Varón,NaN,Sin determinar,Sin determinar,Sin determinación,NaN,Sin relación
4,11536,Imputado idRegistro 12933,Imputado,2,1,No,58,Neuquén,58035,Confluencia,...,---------,No corresponde,Masculino,Varón,NaN,20-24,Sí,Civil,NaN,Sin relación


## Conocer información básica

In [6]:
print('Cantidad de filas y columnas', df.shape)
print('Nombre columnas', df.columns)

Cantidad de filas y columnas (42051, 46)
Nombre columnas Index(['id_hecho', 'tipo_persona_id', 'tipo_persona', 'cant_inc', 'cant_vic',
       'federal', 'provincia_id', 'provincia_nombre', 'departamento_id',
       'departamento_nombre', 'localidad_id', 'localidad_nombre',
       'radio_censal', 'latitud_radio', 'longitud_radio', 'anio', 'mes',
       'fecha_hecho', 'hora_hecho', 'tipo_lugar', 'tipo_lugar_otro',
       'tipo_lugar_ampliado', 'clase_arma', 'clase_arma_otro',
       'en_ocasion_otro_delito', 'en_ocasion_otro_delito_otro',
       'motivo_origen_registro', 'motivo_origen_registro_otro', 'victima_sexo',
       'victima_identidad_genero', 'victima_identidad_genero_otro',
       'victima_tr_edad', 'victima_18_años_o_mas', 'victima_clase',
       'victima_clase_otro', 'victima_situacion_ocupacional',
       'victi_situacion_ocupacional_otro', 'victima_relacion_inculpado',
       'inculpado_sexo', 'inculpado_identidad_genero',
       'inculpado_identidad_genero_otro', 'inculpad

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42051 entries, 0 to 42050
Data columns (total 46 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   id_hecho                          42051 non-null  int64  
 1   tipo_persona_id                   42051 non-null  object 
 2   tipo_persona                      42051 non-null  object 
 3   cant_inc                          42051 non-null  int64  
 4   cant_vic                          42051 non-null  int64  
 5   federal                           42051 non-null  object 
 6   provincia_id                      42051 non-null  int64  
 7   provincia_nombre                  42051 non-null  object 
 8   departamento_id                   42051 non-null  int64  
 9   departamento_nombre               42051 non-null  object 
 10  localidad_id                      42045 non-null  object 
 11  localidad_nombre                  42051 non-null  object 
 12  radi

## Seleccionar solo las variables que voy a utilizar

In [8]:
# selecciona columnas que se van a usar para el análisis
# Selecciona las columnas por su nombre, incluyendo las que faltaban
columnas = [
    'id_hecho',
    'tipo_persona_id',
    'provincia_id', 
    'provincia_nombre', 
    'localidad_id', 
    'localidad_nombre', 
    'anio', 
    'mes', 
    'fecha_hecho', 
    'hora_hecho', 
    'tipo_lugar', 
    'tipo_lugar_otro', 
    'tipo_lugar_ampliado', 
    'motivo_origen_registro', 
    'motivo_origen_registro_otro', 
    'victima_sexo', 
    'victima_identidad_genero', 
    'victima_identidad_genero_otro', 
    'victima_tr_edad', 
    'victima_18_años_o_mas', 
    'inculpado_sexo', 
    'inculpado_identidad_genero', 
    'inculpado_identidad_genero_otro',
    'inculpado_tr_edad',
    'inculpado_18_años_o_mas'
]

df_nuevo = df[columnas]

### Filtrar el dataset para conservar únicamente a las víctimas

**El problema:** 
El dataset crudo no registra "un incidente por fila", sino "una persona involucrada por fila". Esto significa que en un mismo homicidio hay filas para las víctimas y filas para los imputados (agresores). Cuando la fila pertenece a un imputado, todas las columnas de la víctima (`victima_sexo`, `victima_tr_edad`, etc.) se rellenan automáticamente con el valor `'No corresponde'`. 

**La solución:** 
Para que nuestro análisis mida el riesgo real y no duplique casos, eliminamos todas las filas donde el sexo de la víctima sea `'No corresponde'`, conservando así solamente a las víctimas reales.

**Nota metodológica importante:** 
Este filtro se debe aplicar **antes** de la etapa de estandarización de textos. Si primero reemplazamos los `'No corresponde'` por `'Sin determinar'` (para limpiar valores nulos), los agresores se camuflarían como "víctimas con datos desconocidos" y sería imposible filtrarlos.

In [9]:
# Filtrar para conservar ÚNICAMENTE a las víctimas reales
# Elimina las filas donde la columna victima_sexo tiene el valor 'No corresponde'
df_nuevo = df_nuevo[df_nuevo['victima_sexo'] != 'No corresponde']

In [10]:
df_nuevo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19089 entries, 0 to 42049
Data columns (total 25 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   id_hecho                         19089 non-null  int64 
 1   tipo_persona_id                  19089 non-null  object
 2   provincia_id                     19089 non-null  int64 
 3   provincia_nombre                 19089 non-null  object
 4   localidad_id                     19086 non-null  object
 5   localidad_nombre                 19089 non-null  object
 6   anio                             19089 non-null  int64 
 7   mes                              19089 non-null  int64 
 8   fecha_hecho                      19089 non-null  object
 9   hora_hecho                       19089 non-null  object
 10  tipo_lugar                       19089 non-null  object
 11  tipo_lugar_otro                  54 non-null     object
 12  tipo_lugar_ampliado              1908

## Limpieza

In [11]:
df_nuevo.dtypes

id_hecho                            int64
tipo_persona_id                    object
provincia_id                        int64
provincia_nombre                   object
localidad_id                       object
localidad_nombre                   object
anio                                int64
mes                                 int64
fecha_hecho                        object
hora_hecho                         object
tipo_lugar                         object
tipo_lugar_otro                    object
tipo_lugar_ampliado                object
motivo_origen_registro             object
motivo_origen_registro_otro        object
victima_sexo                       object
victima_identidad_genero           object
victima_identidad_genero_otro      object
victima_tr_edad                    object
victima_18_años_o_mas              object
inculpado_sexo                     object
inculpado_identidad_genero         object
inculpado_identidad_genero_otro    object
inculpado_tr_edad                 

### Corregir el formato de hora y fecha

In [12]:
# Convertir a datatime (errors='coerse' convierte errores en nulos)
df_nuevo = df_nuevo.copy()
df_nuevo['fecha_hecho'] = pd.to_datetime(df_nuevo['fecha_hecho'], format='%d/%m/%Y', errors='coerce')
df_nuevo['hora_hecho'] = pd.to_datetime(df_nuevo['hora_hecho'],format='%H:%M:%S', errors='coerce').dt.time

### Detectar NaN

In [13]:
# buscar nulos
df_nuevo.isnull().sum()

id_hecho                               0
tipo_persona_id                        0
provincia_id                           0
provincia_nombre                       0
localidad_id                           3
localidad_nombre                       0
anio                                   0
mes                                    0
fecha_hecho                            0
hora_hecho                             0
tipo_lugar                             0
tipo_lugar_otro                    19035
tipo_lugar_ampliado                    0
motivo_origen_registro                 0
motivo_origen_registro_otro        19022
victima_sexo                           0
victima_identidad_genero               0
victima_identidad_genero_otro      19089
victima_tr_edad                        0
victima_18_años_o_mas                  0
inculpado_sexo                         0
inculpado_identidad_genero             0
inculpado_identidad_genero_otro     5616
inculpado_tr_edad                      0
inculpado_18_año

In [14]:
# tratar valores nulos y rellenar con texto
df_nuevo = df_nuevo.dropna(subset=['localidad_id']) # no es relevante la cantidad de NaN
df_nuevo['tipo_lugar_otro'] = df_nuevo['tipo_lugar_otro'].fillna('No especificado')
df_nuevo['motivo_origen_registro_otro'] = df_nuevo['motivo_origen_registro_otro'].fillna('No especificado')
df_nuevo['victima_identidad_genero_otro'] = df_nuevo['victima_identidad_genero_otro'].fillna('No especificado')
df_nuevo['inculpado_identidad_genero_otro'] = df_nuevo['inculpado_identidad_genero_otro'].fillna('No especificado')

###  Limpiar y estandarizar textos

In [15]:
# detectar errores de escritura, categorías y desbalances
for col in df_nuevo.select_dtypes(include='object').columns:
    print(df_nuevo[col].value_counts(dropna=False))


tipo_persona_id
Víctima idRegistro 44783    1
Víctima idRegistro 12235    1
Víctima idRegistro 12236    1
Víctima idRegistro 12237    1
Víctima idRegistro 13009    1
                           ..
Víctima idRegistro 12277    1
Víctima idRegistro 12276    1
Víctima idRegistro 12275    1
Víctima idRegistro 26909    1
Víctima idRegistro 12251    1
Name: count, Length: 19086, dtype: int64
provincia_nombre
Buenos Aires                       7818
Santa Fe                           2927
Ciudad Autónoma de Buenos Aires    1027
Córdoba                             982
Tucumán                             929
Mendoza                             748
Salta                               620
Chaco                               573
Misiones                            466
Entre Ríos                          446
Neuquén                             318
Santiago del Estero                 316
Chubut                              316
Corrientes                          301
Formosa                             

#### Tratamos 'tipo_persona_id'

In [16]:
# Mostrar los valores únicos de las columnas
df_nuevo['tipo_persona_id'].unique()

array(['Víctima idRegistro 12235', 'Víctima idRegistro 12236',
       'Víctima idRegistro 12237', ..., 'Víctima idRegistro 44856',
       'Víctima idRegistro 44857', 'Víctima idRegistro 44783'],
      shape=(19086,), dtype=object)

In [17]:
# Cuenta la cantidad de valores únicos, sin contar los valores nulos.
df_nuevo['tipo_persona_id'].nunique()

19086

In [18]:
# Quitamos el texto y convertimos el ID a número entero
df_nuevo['tipo_persona_id'] = df_nuevo['tipo_persona_id'].str.replace('Víctima idRegistro ', '').astype('int64')

In [19]:
# Verificamos cómo quedó
df_nuevo['tipo_persona_id'].head()

0     12235
6     12236
7     12237
10    13009
12    12239
Name: tipo_persona_id, dtype: int64

In [20]:
df_nuevo['tipo_persona_id'].unique()

array([12235, 12236, 12237, ..., 44856, 44857, 44783], shape=(19086,))

#### Tratamos 'localidad_id'

In [21]:
# Mostrar los valores únicos de las columnas
df_nuevo['localidad_id'].unique()

array(['100', '40', '70', '10', '30', '130', '50', '220', '20', '200',
       '170', '310', '60', '270', '180', '80', '230', '160', '350', '120',
       '110', '105030', '21010', '35020', '210', '90', '21020', '999',
       '140', '260', '150', '450', '390', '400', '250', '190', '240',
       '360', '112020', '35010', '320', '560', '999-999-2', '280', '370',
       '140020', '999-999-6', '290', '300', '28010', '126070', '340',
       '70020', '440', '154040', '77020', '126060', '330', '380', '550',
       '460', '480'], dtype=object)

In [22]:
# Cuenta la cantidad de valores únicos, sin contar los valores nulos.
df_nuevo['localidad_id'].unique()

array(['100', '40', '70', '10', '30', '130', '50', '220', '20', '200',
       '170', '310', '60', '270', '180', '80', '230', '160', '350', '120',
       '110', '105030', '21010', '35020', '210', '90', '21020', '999',
       '140', '260', '150', '450', '390', '400', '250', '190', '240',
       '360', '112020', '35010', '320', '560', '999-999-2', '280', '370',
       '140020', '999-999-6', '290', '300', '28010', '126070', '340',
       '70020', '440', '154040', '77020', '126060', '330', '380', '550',
       '460', '480'], dtype=object)

In [23]:
# Mostrar los valores únicos con su representación exacta
# para ver diferencias entre texto, números, espacios en blanco, etc.
#for x in df_nuevo['localidad_id'].unique():
   # print(repr(x))

In [24]:
# Mostrar las filas donde es igual a, para ver si la info es relevante
print(df_nuevo[df_nuevo['localidad_id'] == '999-999-2'])
print(df_nuevo[df_nuevo['localidad_id'] == '999-999-6'])

       id_hecho  tipo_persona_id  provincia_id  \
6516      16185            17081             2   
15935     21996            23217             2   

                      provincia_nombre localidad_id          localidad_nombre  \
6516   Ciudad Autónoma de Buenos Aires    999-999-2  Localidad sin determinar   
15935  Ciudad Autónoma de Buenos Aires    999-999-2  Localidad sin determinar   

       anio  mes fecha_hecho hora_hecho  ... victima_sexo  \
6516   2018   11  2018-11-16   21:45:00  ...    Masculino   
15935  2020    8  2020-08-15   06:15:00  ...    Masculino   

      victima_identidad_genero victima_identidad_genero_otro victima_tr_edad  \
6516                     Varón               No especificado           25-29   
15935                    Varón               No especificado           15-19   

      victima_18_años_o_mas  inculpado_sexo inculpado_identidad_genero  \
6516                     Sí  No corresponde             No corresponde   
15935                    No  No 

Se detectaron códigos especiales (999-999-2 y 999-999-6) cuyo contenido en 'localidad_nombre' 
es 'Localidad sin determinar'.
Como el análisis busca mostrar cuáles son las tres provincias y localidades
donde se han detectado la mayor cantidad de homicidios,
se decide reemplazar dichos valores por nulos.

In [25]:
# Reemplazar los códigos de "Localidad sin determinar" por valores nulos (NaN)
df_nuevo['localidad_id'] = df_nuevo['localidad_id'].replace(['999-999-2', '999-999-6'], pd.NA)

In [26]:
# Convertir la columna a tipo numérico (los valores no convertibles quedan como NaN)
df_nuevo['localidad_id'] = pd.to_numeric(df_nuevo['localidad_id'], errors='coerce').astype('Int64')

In [27]:
# Verificar que la conversión se realizó correctamente
df_nuevo['localidad_id'].dtype

Int64Dtype()

In [28]:
df_nuevo['localidad_id'].unique()

<IntegerArray>
[   100,     40,     70,     10,     30,    130,     50,    220,     20,
    200,    170,    310,     60,    270,    180,     80,    230,    160,
    350,    120,    110, 105030,  21010,  35020,    210,     90,  21020,
    999,    140,    260,    150,    450,    390,    400,    250,    190,
    240,    360, 112020,  35010,    320,    560,   <NA>,    280,    370,
 140020,    290,    300,  28010, 126070,    340,  70020,    440, 154040,
  77020, 126060,    330,    380,    550,    460,    480]
Length: 61, dtype: Int64

In [29]:
df_nuevo['localidad_id'].nunique()

60

### Tratamos 'hora_hecho'

In [30]:
# cuenta cuántas veces aparece cada valor de la columna hora_hecho
# ordenándolos de mayor a menor frecuencia.
df_nuevo['hora_hecho'].value_counts()

hora_hecho
11:11:11    6944
11:11:00     408
00:00:00     293
22:00:00     240
21:00:00     226
            ... 
20:31:00       1
14:31:00       1
05:16:00       1
08:29:00       1
02:38:00       1
Name: count, Length: 1152, dtype: int64

In [31]:
# Filtra los registros cuya hora_hecho sea exactamente las 11:11:11.
from datetime import time
df_nuevo[df_nuevo['hora_hecho'] == time(11,11,11)]

,id_hecho,tipo_persona_id,provincia_id,provincia_nombre,localidad_id,localidad_nombre,anio,mes,fecha_hecho,hora_hecho,...,victima_sexo,victima_identidad_genero,victima_identidad_genero_otro,victima_tr_edad,victima_18_años_o_mas,inculpado_sexo,inculpado_identidad_genero,inculpado_identidad_genero_otro,inculpado_tr_edad,inculpado_18_años_o_mas
773,12038,12759,82,Santa Fe,270,Rosario,2017,8,2017-08-31,11:11:11,...,Masculino,Varón,No especificado,65-69,Sí,No corresponde,No corresponde,-------,No corresponde,No corresponde
1837,12710,13523,82,Santa Fe,270,Rosario,2018,1,2018-01-28,11:11:11,...,Masculino,Varón,No especificado,85-89,Sí,No corresponde,No corresponde,-------,No corresponde,No corresponde
2222,12955,13802,62,Río Negro,90,Viedma,2017,1,2017-01-01,11:11:11,...,Masculino,Varón,No especificado,35-39,Sí,No corresponde,No corresponde,-------,No corresponde,No corresponde
2224,12956,13803,62,Río Negro,90,Viedma,2017,1,2017-01-01,11:11:11,...,Masculino,Varón,No especificado,15-19,No,No corresponde,No corresponde,-------,No corresponde,No corresponde
2225,12957,13804,62,Río Negro,450,General Roca,2017,1,2017-01-08,11:11:11,...,Masculino,Varón,No especificado,35-39,Sí,No corresponde,No corresponde,-------,No corresponde,No corresponde
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38101,37743,40007,6,Buenos Aires,10,San Isidro,2024,9,2024-09-24,11:11:11,...,Masculino,Varón,No especificado,15-19,No,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
38103,37744,40008,6,Buenos Aires,50,San Nicolás de los Arroyos,2024,9,2024-09-13,11:11:11,...,Femenino,Mujer,No especificado,25-29,Si,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
38107,37745,40009,6,Buenos Aires,20,Suipacha,2024,9,2024-09-28,11:11:11,...,Masculino,Varón,No especificado,35-39,Si,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
38109,37746,40010,6,Buenos Aires,10,Tigre,2024,9,2024-09-28,11:11:11,...,Masculino,Varón,No especificado,30-34,Si,No corresponde,No corresponde,No especificado,No corresponde,No corresponde


In [32]:
df_nuevo[df_nuevo['hora_hecho'] == time(11,11,00)]

,id_hecho,tipo_persona_id,provincia_id,provincia_nombre,localidad_id,localidad_nombre,anio,mes,fecha_hecho,hora_hecho,...,victima_sexo,victima_identidad_genero,victima_identidad_genero_otro,victima_tr_edad,victima_18_años_o_mas,inculpado_sexo,inculpado_identidad_genero,inculpado_identidad_genero_otro,inculpado_tr_edad,inculpado_18_años_o_mas
29817,30728,32791,58,Neuquén,110,Senillosa,2023,8,2023-08-26,11:11:00,...,Masculino,Varón,No especificado,25-29,Sí,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
30070,30904,32969,82,Santa Fe,270,Rosario,2023,3,2023-03-28,11:11:00,...,Femenino,Mujer,No especificado,25-29,Sí,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
30305,31052,33149,82,Santa Fe,270,Rosario,2023,7,2023-07-11,11:11:00,...,Femenino,Mujer,No especificado,4 o menos,No,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
30315,31057,33154,82,Santa Fe,270,Rosario,2023,7,2023-07-23,11:11:00,...,Masculino,Varón,No especificado,25-29,Sí,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
30458,31143,33241,82,Santa Fe,50,Casilda,2023,8,2023-08-21,11:11:00,...,Masculino,Varón,No especificado,65-69,Sí,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41466,41085,43542,6,Buenos Aires,10,Quilmes,2025,12,2025-12-03,11:11:00,...,Masculino,Varón,No especificado,65-69,Si,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
41468,41086,43543,6,Buenos Aires,10,Quilmes,2025,12,2025-12-24,11:11:00,...,Masculino,Varón,No especificado,20-24,Si,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
41470,41087,43544,6,Buenos Aires,10,Quilmes,2025,12,2025-12-24,11:11:00,...,Masculino,Varón,No especificado,30-34,Si,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
41479,41090,43548,6,Buenos Aires,10,Quilmes,2025,12,2025-12-30,11:11:00,...,Masculino,Varón,No especificado,15-19,No,No corresponde,No corresponde,No especificado,No corresponde,No corresponde


Se conservan los datos con horas '11:11:11' y '11:11:00' ya que representan una gran cantidad de registros necesarios para el análisis. Sin embargo en caso de analizar la hora del suceso, se procederá a tomar dicha hora dentro de una franja horaria más amplia para evitar el impacto que estos valores puedan tener sobre los resultados.  

### Tratamos 'victima_18_años_o_mas'

In [33]:
# ver los valores únicos y las veces que aparecen
df_nuevo['victima_18_años_o_mas'].value_counts(dropna=False)

victima_18_años_o_mas
Sí                13939
Si                 3300
No                 1520
Sin determinar      327
Name: count, dtype: int64

In [34]:
# renombramos la columna
df_nuevo = df_nuevo.rename(columns={'victima_18_años_o_mas':'victima_18_anios_o_mas'})

In [35]:
# unificamos la categoría 'Si' para evitar duplicados
df_nuevo['victima_18_anios_o_mas'] = df_nuevo['victima_18_anios_o_mas'].replace('Sí','Si')


In [36]:
df_nuevo['victima_18_anios_o_mas'].value_counts()

victima_18_anios_o_mas
Si                17239
No                 1520
Sin determinar      327
Name: count, dtype: int64

### Tratamos 'victima_identidad_genero_otro'

In [37]:
df_nuevo['victima_identidad_genero_otro'].value_counts(dropna=False)

victima_identidad_genero_otro
No especificado    19086
Name: count, dtype: int64

### Tratamos ''inculpado_identidad_genero_otro'

In [38]:
df_nuevo['inculpado_identidad_genero_otro'].value_counts(dropna=False)

inculpado_identidad_genero_otro
-------            13473
No especificado     5613
Name: count, dtype: int64

In [39]:
df_nuevo['inculpado_identidad_genero_otro'] = df_nuevo['inculpado_identidad_genero_otro'].replace(
    '-------','No especificado')

In [40]:
df_nuevo['inculpado_identidad_genero_otro'].value_counts()

inculpado_identidad_genero_otro
No especificado    19086
Name: count, dtype: int64

### Tratamos 'tipo_lugar_ampliado'

In [41]:
df_nuevo['tipo_lugar_ampliado'].value_counts(dropna=False)

tipo_lugar_ampliado
Vía pública                                            10344
Domicilio particular                                    5654
Sin determinar                                          1529
Comercio                                                 411
Interior de rodados                                      275
Campo/descampado/zona rural                              204
Cárcel o comisaría                                       167
Establecimiento zona rural                                71
Otro lugar                                                63
Río/canal/arroyo/mar/dique                                54
Basural/centro de reciclaje/relleno sanitario             40
Establecimiento de salud                                  38
Canchas/club                                              37
Bar/ Boliche/Salón de fiestas/Fiesta                      36
Hotel/Motel/Hospedaje temporario                          30
Casa/edificio abandonado /Obra en construcción            27
Bald

In [42]:
df_nuevo['tipo_lugar_ampliado'] = df_nuevo['tipo_lugar_ampliado'].replace(
    'Bar/ Boliche/Salón de fiestas/Fiesta','Bar/Boliche/Salón de fiestas/Fiesta')

In [43]:
df_nuevo['tipo_lugar_ampliado'] = df_nuevo['tipo_lugar_ampliado'].replace(
    'Casa/edificio abandonado /Obra en construcción','Casa/edificio abandonado/Obra en construcción')

In [44]:
df_nuevo['tipo_lugar_ampliado'].value_counts()

tipo_lugar_ampliado
Vía pública                                            10344
Domicilio particular                                    5654
Sin determinar                                          1529
Comercio                                                 411
Interior de rodados                                      275
Campo/descampado/zona rural                              204
Cárcel o comisaría                                       167
Establecimiento zona rural                                71
Otro lugar                                                63
Río/canal/arroyo/mar/dique                                54
Bar/Boliche/Salón de fiestas/Fiesta                       42
Basural/centro de reciclaje/relleno sanitario             40
Establecimiento de salud                                  38
Canchas/club                                              37
Casa/edificio abandonado/Obra en construcción             33
Hotel/Motel/Hospedaje temporario                          30
Bald

## Duplicados

In [45]:
# contar cuántos duplicados hay
df_nuevo.duplicated().sum()

np.int64(0)

In [46]:
# ver cuáles son
#df_nuevo[df_nuevo.duplicated(keep=False)].head()

In [47]:
#df_nuevo[df_nuevo.duplicated(keep=False)].tail()

In [48]:
# Ver qué pasa con esos casos en el dataframe original (df)
#df.loc[[1333, 1334, 1814, 1816, 41323, 41324,41930, 41932], ['id_hecho', 'tipo_persona_id', 'cant_vic', 'victima_sexo', 'victima_tr_edad']]

En una primera instancia de verificación, nos figuraba 107 duplicados. El codigo comentado arriba, fueron los pasos que se tomaron para verificar y que esos 107 casos marcados por duplicated() no son errores de carga, sino víctimas distintas de homicidios múltiples que coincidían en edad y sexo dentro del mismo hecho.
Se solucionó agregando la columna 'tipo_persona_id', eliminando el texto que tenía y dejandolo solo como valor numérico. 

In [49]:
# eliminamos duplicados
# df_nuevo = df_nuevo.drop_duplicates()

In [50]:
# confirmar que no queden duplicados
# df_nuevo.duplicated().sum()

In [51]:
# ver el tamaño final de tabla
print('Cantidad de filas y columnas', df_nuevo.shape)
print('Nombre columnas', df_nuevo.columns)

Cantidad de filas y columnas (19086, 25)
Nombre columnas Index(['id_hecho', 'tipo_persona_id', 'provincia_id', 'provincia_nombre',
       'localidad_id', 'localidad_nombre', 'anio', 'mes', 'fecha_hecho',
       'hora_hecho', 'tipo_lugar', 'tipo_lugar_otro', 'tipo_lugar_ampliado',
       'motivo_origen_registro', 'motivo_origen_registro_otro', 'victima_sexo',
       'victima_identidad_genero', 'victima_identidad_genero_otro',
       'victima_tr_edad', 'victima_18_anios_o_mas', 'inculpado_sexo',
       'inculpado_identidad_genero', 'inculpado_identidad_genero_otro',
       'inculpado_tr_edad', 'inculpado_18_años_o_mas'],
      dtype='object')


### Tratamos 'victima_tr_edad'

In [52]:
df_nuevo['victima_tr_edad'].value_counts()

victima_tr_edad
20-24             2908
25-29             2776
30-34             2580
15-19             1980
35-39             1804
40-44             1473
45-49             1089
50-54              795
55-59              661
Sin determinar     527
60-64              522
65-69              455
70-74              361
4 o menos          263
75-79              259
10-14              203
80-84              194
5-9                 99
85-89               92
90 y más            45
Name: count, dtype: int64

In [53]:
# Filtra y muestra los registros donde tiene el valor 'Sin determinar'.
df_nuevo[df_nuevo['victima_tr_edad'] == 'Sin determinar']

,id_hecho,tipo_persona_id,provincia_id,provincia_nombre,localidad_id,localidad_nombre,anio,mes,fecha_hecho,hora_hecho,...,victima_sexo,victima_identidad_genero,victima_identidad_genero_otro,victima_tr_edad,victima_18_anios_o_mas,inculpado_sexo,inculpado_identidad_genero,inculpado_identidad_genero_otro,inculpado_tr_edad,inculpado_18_años_o_mas
58,11578,12768,66,Salta,70,San Ramón de la Nueva Orán (Est. Orán),2017,1,2017-01-25,01:00:00,...,Masculino,Varón,No especificado,Sin determinar,Sin determinar,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
66,11584,12292,26,Chubut,40,Trelew,2017,2,2017-02-24,19:40:00,...,Masculino,Varón,No especificado,Sin determinar,Si,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
400,11779,12487,66,Salta,10,El Galpón (Est. Foguista J. F. Juárez),2017,3,2017-03-17,00:00:00,...,Femenino,Mujer,No especificado,Sin determinar,Sin determinar,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
576,11921,12639,58,Neuquén,10,San Martín de los Andes,2017,8,2017-08-04,23:55:00,...,Masculino,Varón,No especificado,Sin determinar,Sin determinar,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
892,12191,12933,82,Santa Fe,30,Las Parejas,2017,10,2017-10-20,05:45:00,...,Femenino,Mujer,No especificado,Sin determinar,Sin determinar,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41866,41636,44074,2,Ciudad Autónoma de Buenos Aires,10,Ciudad de Buenos Aires,2025,12,2025-12-28,14:00:00,...,Masculino,Varón,No especificado,Sin determinar,Si,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
41948,42141,44578,2,Ciudad Autónoma de Buenos Aires,10,Ciudad de Buenos Aires,2024,12,2024-12-18,05:00:00,...,Masculino,Varón,No especificado,Sin determinar,Si,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
41958,42147,44584,2,Ciudad Autónoma de Buenos Aires,10,Ciudad de Buenos Aires,2024,7,2024-07-05,22:00:00,...,Masculino,Varón,No especificado,Sin determinar,Si,No corresponde,No corresponde,No especificado,No corresponde,No corresponde
41976,42156,44593,2,Ciudad Autónoma de Buenos Aires,10,Ciudad de Buenos Aires,2024,2,2024-02-09,05:00:00,...,Masculino,Varón,No especificado,Sin determinar,Si,No corresponde,No corresponde,No especificado,No corresponde,No corresponde


### Tratamos 'inculpado_18_años_o_mas'

In [54]:
# renombramos la columna del inculpado
df_nuevo = df_nuevo.rename(columns={'inculpado_18_años_o_mas':'inculpado_18_anios_o_mas'})

In [55]:
# ver los valores únicos y las veces que aparecen
df_nuevo['inculpado_18_anios_o_mas'].value_counts(dropna=False)

inculpado_18_anios_o_mas
No corresponde    19086
Name: count, dtype: int64

### Tratamos 'inculpado_tr_edad'

In [56]:
# ver los valores únicos y las veces que aparecen
df_nuevo['inculpado_tr_edad'].value_counts(dropna=False)

inculpado_tr_edad
No corresponde    19086
Name: count, dtype: int64

In [57]:
df_nuevo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19086 entries, 0 to 42049
Data columns (total 25 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   id_hecho                         19086 non-null  int64         
 1   tipo_persona_id                  19086 non-null  int64         
 2   provincia_id                     19086 non-null  int64         
 3   provincia_nombre                 19086 non-null  object        
 4   localidad_id                     19083 non-null  Int64         
 5   localidad_nombre                 19086 non-null  object        
 6   anio                             19086 non-null  int64         
 7   mes                              19086 non-null  int64         
 8   fecha_hecho                      19086 non-null  datetime64[ns]
 9   hora_hecho                       19086 non-null  object        
 10  tipo_lugar                       19086 non-null  object        

In [58]:
df_nuevo.to_csv('homicidios_limpio.csv', index=False, sep=';', encoding='utf-8')